In [1]:
!python --version
!nvidia-smi

Python 3.12.13
Wed Sep  2 11:32:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------------

In [2]:
!pip install -q uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 89.9 MB/s eta 0:00:00


In [3]:
import subprocess, sys

VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

In [4]:
pip_install(
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
)

print("serving pins installed")


installing: vllm==0.6.* transformers==4.46.* accelerate==1.1.* httpx==0.27.* openai==1.54.*
serving pins installed


In [5]:
import os, signal, subprocess

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
SERVER_LOG = "/content/server.log"


SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--enforce-eager": None,
    "--port": str(PORT),
}

def build_cmd(args: dict) -> list:
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)

    print("launching:", " ".join(cmd))

    logf = open(SERVER_LOG, "wb")

    proc = subprocess.Popen(
        cmd,
        stdout=logf,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server()

launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --enforce-eager --port 8000
server pid 2460, logging to /content/server.log


In [6]:
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s

    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True

        except (urllib.error.URLError, ConnectionError, OSError):
            pass

        time.sleep(interval_s)

    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    return False

healthy = wait_for_health()

server healthy after about 133s: http://localhost:8000/v1/models -> 200


In [7]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

r = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "In one sentence, what is a GPU?"
        }
    ],
)

print(r.choices[0].message.content)

A GPU, or Graphics Processing Unit, is a specialized processor designed to accelerate computations involved in rendering graphics and video content on electronic devices.


In [8]:
from google.colab import files

uploaded = files.upload()   # choose baselines.json

import json

baseline = json.load(open("baselines.json"))

print("baseline batch tokens/s:", baseline["batch"])

Saving baselines.json to baselines.json
baseline batch tokens/s: {'1': 34.7, '4': 52.6, '8': 104.9}


In [9]:
%run ab_client.py

In [10]:
prompts = FIXED_PROMPTS

vllm_measured = await run_sweep(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompts=prompts,
    concurrencies=[1, 4, 8],
)

for level in vllm_measured:
    print(level)

level: {'concurrency': 1, 'requests': 24, 'tokens_per_s': 36.0, 'wall_s': 38.628}
level: {'concurrency': 4, 'requests': 24, 'tokens_per_s': 101.9, 'wall_s': 13.628}
level: {'concurrency': 8, 'requests': 24, 'tokens_per_s': 167.4, 'wall_s': 8.297}
{'concurrency': 1, 'requests': 24, 'tokens_per_s': 36.0, 'wall_s': 38.628}
{'concurrency': 4, 'requests': 24, 'tokens_per_s': 101.9, 'wall_s': 13.628}
{'concurrency': 8, 'requests': 24, 'tokens_per_s': 167.4, 'wall_s': 8.297}


In [13]:
import json

def tokps_at(level_list, c):
    return next(
        x["tokens_per_s"]
        for x in level_list
        if x["concurrency"] == c
    )

vllm_by_c = {
    x["concurrency"]: x["tokens_per_s"]
    for x in vllm_measured
}

base_by_c = {
    int(k): v
    for k, v in baseline["batch"].items()
}

speedup = {
    c: round(vllm_by_c[c] / base_by_c[c], 2)
    for c in vllm_by_c
    if c in base_by_c
}

report = {
    "baseline": base_by_c,
    "vllm": vllm_by_c,
    "speedup_by_concurrency": speedup,
    "predicted_speedup": 1.5,
}

with open("ab_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

{
  "baseline": {
    "1": 34.7,
    "4": 52.6,
    "8": 104.9
  },
  "vllm": {
    "1": 36.0,
    "4": 101.9,
    "8": 167.4
  },
  "speedup_by_concurrency": {
    "1": 1.04,
    "4": 1.94,
    "8": 1.6
  },
  "predicted_speedup": 1.5
}


In [14]:
static_scaling = base_by_c[8] / base_by_c[1]
vllm_scaling   = vllm_by_c[8] / vllm_by_c[1]

print(f"static batching scales {static_scaling:.2f}x, vLLM scales {vllm_scaling:.2f}x")
print(f"continuous batching is worth {vllm_scaling / static_scaling:.2f}x of scaling")

static batching scales 3.02x, vLLM scales 4.65x
continuous batching is worth 1.54x of scaling


In [15]:
%run verify_cell.py

baseline batch-8: 104.9, vllm concurrency-8: 167.4
speedup at 8: 1.6x
GREEN CHECK: PASS




---





---



---



In [16]:
from google.colab import files
files.download("ab_report.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
import asyncio, time, httpx

async def send_one(client, base_url, model, prompt, max_tokens=128):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.0
    }

    t0 = time.perf_counter()

    try:
        r = await client.post(
            f"{base_url}/chat/completions",
            json=payload
        )
        r.raise_for_status()

        return {
            "ok": True,
            "latency_s": time.perf_counter() - t0
        }

    except Exception as e:
        return {
            "ok": False,
            "latency_s": time.perf_counter() - t0,
            "error": str(e)
        }


def p95(latencies):
    s = sorted(latencies)
    idx = max(0, int(len(s) * 0.95) - 1)
    return s[idx]


async def naive_burst(base_url, model, prompt, n=50, max_tokens=128):
    async with httpx.AsyncClient(timeout=120.0) as client:

        results = await asyncio.gather(*[
            send_one(
                client,
                base_url,
                model,
                prompt,
                max_tokens
            )
            for _ in range(n)
        ])

    latencies = [
        r["latency_s"]
        for r in results
        if r["ok"]
    ]

    return {
        "n_sent": n,
        "n_ok": len(latencies),
        "p95_s": round(p95(latencies), 3) if latencies else None,
        "mean_s": round(sum(latencies) / len(latencies), 3)
        if latencies else None,
    }


prompt = "In two sentences, explain what a load balancer does."

naive_result = await naive_burst(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompt=prompt,
    n=50,
)

print("naive (unbounded):", naive_result)

naive (unbounded): {'n_sent': 50, 'n_ok': 50, 'p95_s': 1.563, 'mean_s': 1.553}


In [18]:
class LoadShedder:
    def __init__(self, max_in_flight: int):
        self.sem = asyncio.Semaphore(max_in_flight)

    async def try_admit(self):
        # Admit only if a slot is free RIGHT NOW
        acquired = self.sem.locked() is False and self.sem._value > 0
        if acquired:
            await self.sem.acquire()
        return acquired

    def release(self):
        self.sem.release()


async def send_with_shedding(
    client, shedder, base_url, model, prompt, max_tokens=128
):
    admitted = await shedder.try_admit()

    if not admitted:
        return {
            "ok": False,
            "shed": True,
            "latency_s": 0.0
        }

    try:
        t0 = time.perf_counter()

        r = await client.post(
            f"{base_url}/v1/chat/completions".replace("/v1/v1", "/v1"),
            json={
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": max_tokens,
                "temperature": 0.0
            }
        )

        r.raise_for_status()

        return {
            "ok": True,
            "shed": False,
            "latency_s": time.perf_counter() - t0
        }

    except Exception as e:
        return {
            "ok": False,
            "shed": False,
            "latency_s": time.perf_counter() - t0,
            "error": str(e)
        }

    finally:
        shedder.release()


async def shedded_burst(
    base_url, model, prompt, n=50, cap=8, max_tokens=128
):
    shedder = LoadShedder(cap)

    async with httpx.AsyncClient(timeout=120.0) as client:
        results = await asyncio.gather(*[
            send_with_shedding(
                client, shedder, base_url,
                model, prompt, max_tokens
            )
            for _ in range(n)
        ])

    accepted = [r for r in results if r["ok"]]
    shed = [r for r in results if r.get("shed")]
    latencies = [r["latency_s"] for r in accepted]

    return {
        "n_sent": n,
        "cap": cap,
        "n_accepted": len(accepted),
        "n_shed": len(shed),
        "accepted_p95_s": round(p95(latencies), 3)
        if latencies else None,
    }


shedded_result = await shedded_burst(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompt=prompt,
    n=50,
    cap=8,
)

print("shedded (cap=8):", shedded_result)

shedded (cap=8): {'n_sent': 50, 'cap': 8, 'n_accepted': 8, 'n_shed': 42, 'accepted_p95_s': 1.053}


In [19]:
sweep = []

for n in (8, 16, 32, 50):
    r = await shedded_burst(
        base_url="http://localhost:8000/v1",
        model="Qwen/Qwen2.5-1.5B-Instruct",
        prompt=prompt,
        n=n,
        cap=8,
    )
    sweep.append(r)
    print(r)

{'n_sent': 8, 'cap': 8, 'n_accepted': 8, 'n_shed': 0, 'accepted_p95_s': 0.761}
{'n_sent': 16, 'cap': 8, 'n_accepted': 8, 'n_shed': 8, 'accepted_p95_s': 0.457}
{'n_sent': 32, 'cap': 8, 'n_accepted': 8, 'n_shed': 24, 'accepted_p95_s': 0.469}
{'n_sent': 50, 'cap': 8, 'n_accepted': 8, 'n_shed': 42, 'accepted_p95_s': 0.432}


In [20]:
import json

report = {
    "naive_unbounded_n50": naive_result,
    "shedded_cap8_n50": shedded_result,
    "shedded_sweep": sweep,
}

with open("shedding_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

{
  "naive_unbounded_n50": {
    "n_sent": 50,
    "n_ok": 50,
    "p95_s": 1.563,
    "mean_s": 1.553
  },
  "shedded_cap8_n50": {
    "n_sent": 50,
    "cap": 8,
    "n_accepted": 8,
    "n_shed": 42,
    "accepted_p95_s": 1.053
  },
  "shedded_sweep": [
    {
      "n_sent": 8,
      "cap": 8,
      "n_accepted": 8,
      "n_shed": 0,
      "accepted_p95_s": 0.761
    },
    {
      "n_sent": 16,
      "cap": 8,
      "n_accepted": 8,
      "n_shed": 8,
      "accepted_p95_s": 0.457
    },
    {
      "n_sent": 32,
      "cap": 8,
      "n_accepted": 8,
      "n_shed": 24,
      "accepted_p95_s": 0.469
    },
    {
      "n_sent": 50,
      "cap": 8,
      "n_accepted": 8,
      "n_shed": 42,
      "accepted_p95_s": 0.432
    }
  ]
}


In [26]:
!python verify.py

invariants hold: shedding happened, accepted p95 protected, cap flat
GREEN CHECK: PASS


In [27]:
from google.colab import files
files.download("shedding_report.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>